# Simple Video RAG with VideoDB and LlamaIndex

This notebook shows how to build a simple video RAG pipeline with VideoDB and LlamaIndex.

We will use VideoDB to:

- Upload two videos.
- Transcribe each video with `understand(spoken_words)`.
- Convert transcript artifacts into timestamped text windows.
- Index those windows with `video.index()`.
- Expose VideoDB search results through a LlamaIndex retriever.
- Generate text answers and playable video streams from retrieved moments.

## 1. Install Dependencies

In [ ]:
!pip install -q videodb python-dotenv llama-index openai


## 2. Connect to VideoDB

Enter your VideoDB API key and OpenAI API key when prompted. LlamaIndex uses OpenAI for response synthesis in this notebook.

In [2]:
import os
import time
from getpass import getpass
from uuid import uuid4

from IPython.display import display
from videodb import connect, play_stream

os.environ["VIDEO_DB_API_KEY"] = getpass("Please enter your VideoDB API Key: ")
os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API Key: ")

conn = connect()
coll = conn.get_collection()

print("Connected to VideoDB successfully.")

Please enter your VideoDB API Key: ··········
Please enter your OpenAI API Key: ··········
Connected to VideoDB successfully.


## 3. Upload Videos

Upload two public videos into the default collection. You can replace these URLs with your own videos.

In [3]:
video_sources = [
    {
        "title": "Dopamine and Motivation",
        "url": "https://www.youtube.com/watch?v=lsODSDmY4CY",
    },
    {
        "title": "Morning Sunlight and Sleep",
        "url": "https://www.youtube.com/watch?v=vZ4kOr38JhY",
    },
]

videos = []

for source in video_sources:
    print("Uploading:", source["title"])
    video = coll.upload(url=source["url"])
    videos.append({"title": source["title"], "video": video})
    print("Video ID:", video.id)

Uploading: Dopamine and Motivation
Video ID: m-z-019f37a7-f68b-7c11-9db2-813cbf3073dc
Uploading: Morning Sunlight and Sleep
Video ID: m-z-019f37ab-399e-7f02-9eb6-c27719185937


## 4. Transcribe Videos

Run `understand(spoken_words)` on each video and collect the transcript artifacts.

In [4]:
def transcribe_video(video):
    understanding = video.understand(
        analyzers=[
            {
                "type": "spoken_words",
                "name": "transcript",
                "config": {
                    "language": "en",
                },
            }
        ],
    )
    understanding.wait_until_complete(timeout=3600, poll_interval=15)
    return understanding.get_analyzer("transcript").get_output()


for item in videos:
    print("Transcribing:", item["title"])
    transcript_output = transcribe_video(item["video"])
    item["transcript_output"] = transcript_output
    print("Status:", transcript_output.get("status"))
    print("Transcript scenes:", len(transcript_output.get("scenes", [])))

Transcribing: Dopamine and Motivation
Status: done
Transcript scenes: 7
Transcribing: Morning Sunlight and Sleep
Status: done
Transcript scenes: 7


## 5. Build Transcript Windows

For RAG, it is useful to index medium-sized transcript windows instead of individual words. Each record keeps `start`, `end`, `text`, and source metadata so retrieved nodes can point back to playable video moments.

In [5]:
def flatten_transcript_words(transcript_output):
    words = []

    for scene in transcript_output.get("scenes", []):
        words.extend(scene.get("data", {}).get("words", []))

    seen = set()
    deduped = []

    for word in words:
        key = (word.get("start"), word.get("end"), word.get("text"))
        if key in seen:
            continue
        seen.add(key)
        deduped.append(word)

    return sorted(deduped, key=lambda word: float(word.get("start", 0)))


def build_transcript_windows(transcript_output, video, title, window_seconds=45):
    words = flatten_transcript_words(transcript_output)
    windows = []
    current_words = []
    current_start = None
    current_end = None

    for word in words:
        if word.get("start") is None or word.get("end") is None or not word.get("text"):
            continue

        word_start = float(word["start"])
        word_end = float(word["end"])

        if current_start is None:
            current_start = word_start

        if current_words and word_start - current_start >= window_seconds:
            windows.append(
                {
                    "start": current_start,
                    "end": current_end,
                    "text": " ".join(current_words).strip(),
                    "video_id": video.id,
                    "source_video_title": title,
                }
            )
            current_words = []
            current_start = word_start

        current_words.append(word["text"])
        current_end = word_end

    if current_words:
        windows.append(
            {
                "start": current_start,
                "end": current_end,
                "text": " ".join(current_words).strip(),
                "video_id": video.id,
                "source_video_title": title,
            }
        )

    return windows


for item in videos:
    item["transcript_windows"] = build_transcript_windows(
        item["transcript_output"],
        item["video"],
        item["title"],
    )
    print(item["title"], "windows:", len(item["transcript_windows"]))
    print(item["transcript_windows"][0])

Dopamine and Motivation windows: 34
{'start': 0.4, 'end': 45.52, 'text': "Welcome to the Huberman Lab podcast where we discuss science and science based tools for. Everyday life. I'm andrew huberman. And I'm a professor of neurobiology and ophthalmology at Stanford School of Medicine today. Is an Ask me Anything or AMA episode, which is part of our premium subscriber. Content. Our premium channel was launched in order to raise support for the standard Huberman Lab podcast. Ch. Channel, which still comes out once a week every Monday, and, of course, is zero cost to consum. The premium channel is also designed to support exciting research being done at major universities. Like Stanford and elsewhere. Research that's done on humans that should lead to protocols. For mental health, physical health and performance in the near future. If you'd like to check out. The premium", 'video_id': 'm-z-019f37a7-f68b-7c11-9db2-813cbf3073dc', 'source_video_title': 'Dopamine and Motivation'}
Morning Sunl

## 6. Index Transcript Windows

Index each video's transcript windows for semantic retrieval. We use unique index names so the notebook can be rerun without name collisions.

In [6]:
INDEX_READY_STATUSES = {"ready", "done"}
INDEX_ACTIVE_STATUSES = {"building", "processing"}


def wait_until_index_ready(video, index, timeout=1800, poll_interval=10):
    deadline = time.time() + timeout
    latest_index = index

    while time.time() < deadline:
        status = getattr(latest_index, "status", None)

        if status in INDEX_READY_STATUSES:
            return latest_index

        if status and status not in INDEX_ACTIVE_STATUSES:
            raise RuntimeError(f"Index build ended with status: {status}")

        time.sleep(poll_interval)

        try:
            latest_index = video.get_index(index_id=index.index_id)
        except Exception as exc:
            if "not found" in str(exc).lower():
                continue
            raise

    raise TimeoutError(f"Index was not ready within {timeout} seconds.")


for item in videos:
    index_name = f"video_rag_transcript_{uuid4().hex[:8]}"
    index = item["video"].index(
        name=index_name,
        source=item["transcript_windows"],
        use_for=["semantic"],
        fields={
            "semantic": ["text"],
            "text": ["text"],
            "filter": ["source_video_title"],
        },
    )
    item["index"] = wait_until_index_ready(item["video"], index)
    item["index_name"] = index_name
    print(item["title"], item["index"].index_id, item["index"].status)

Dopamine and Motivation b239acb81de74d57 ready
Morning Sunlight and Sleep c213ec50c46a4e00 ready


## 7. Build a LlamaIndex Retriever

This small retriever adapts VideoDB semantic results into LlamaIndex `TextNode` objects. Each node keeps video metadata so you can later stream the exact timestamps that supported the answer.

In [7]:
from llama_index.core import get_response_synthesizer
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore, TextNode


class VideoDBSemanticRetriever(BaseRetriever):
    def __init__(self, indexed_videos, top_k=3):
        super().__init__()
        self.indexed_videos = indexed_videos
        self.top_k = top_k

    def _retrieve(self, query_bundle):
        query = query_bundle.query_str
        nodes = []

        for item in self.indexed_videos:
            results = item["video"].semantic_search(
                query=query,
                index_names=[item["index_name"]],
                top_k=self.top_k,
                return_fields=["text", "source_video_title"],
            )

            for shot in results.shots:
                metadata = getattr(shot, "metadata", {}) or {}
                text = metadata.get("text", "")
                node = TextNode(
                    text=text,
                    metadata={
                        "video_id": item["video"].id,
                        "title": item["title"],
                        "start": shot.start,
                        "end": shot.end,
                    },
                )
                score = getattr(shot, "score", None)
                nodes.append(NodeWithScore(node=node, score=score))

        return nodes[: self.top_k]


retriever = VideoDBSemanticRetriever(videos, top_k=4)
response_synthesizer = get_response_synthesizer()
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

print("LlamaIndex query engine is ready.")

LlamaIndex query engine is ready.


## 8. Ask Questions Across Videos

Ask natural-language questions. LlamaIndex synthesizes an answer from VideoDB-retrieved transcript windows.

In [8]:
response = query_engine.query("What is dopamine and why is it important?")
print(response)

Dopamine is a neurotransmitter that plays a key role in motivation, reward, and pleasure. It is important because it helps regulate mood, behavior, focus, and feelings of pleasure and satisfaction. Dopamine is also involved in various brain functions such as movement, memory, and learning.


In [9]:
response = query_engine.query("What is the benefit of morning sunlight?")
print(response)

Morning sunlight can help regulate the body's internal clock, improve mood, boost Vitamin D levels, and enhance alertness and productivity throughout the day.


## 9. Inspect Retrieved Nodes

Retrieve the source nodes directly to see the exact video IDs and timestamps that support the answer.

In [10]:
question = "What is the benefit of morning sunlight?"
relevant_nodes = retriever.retrieve(question)

for node_with_score in relevant_nodes:
    node = node_with_score.node
    print(node.metadata["title"])
    print(f"{node.metadata['start']:.2f}s - {node.metadata['end']:.2f}s")
    print(node.text[:500])
    print("----")

Dopamine and Motivation
542.40s - 587.60s

----
Dopamine and Motivation
903.52s - 948.68s

----
Dopamine and Motivation
948.68s - 994.20s

----
Dopamine and Motivation
723.12s - 768.20s

----


## 10. Watch Retrieved Moments

Compile the retrieved moments into one playable stream.

In [18]:
from videodb.editor import Timeline, Track, Clip, VideoAsset

timeline = Timeline(conn)
track = Track()

current_time = 0

for node_with_score in relevant_nodes:
    metadata = node_with_score.node.metadata
    start = float(metadata["start"])
    end = float(metadata["end"])
    duration = max(end - start, 0.1)

    video_asset = VideoAsset(
        id=metadata["video_id"],
        start=start,
    )

    video_clip = Clip(
        asset=video_asset,
        duration=duration,
    )

    track.add_clip(current_time, video_clip)
    current_time += duration

timeline.add_track(track)

stream_url = timeline.generate_stream()

player = play_stream(stream_url)
display(player)

## 11. Optional Cleanup

Set `DELETE_UPLOADED_VIDEOS = True` if you want to remove the uploaded sample videos after testing.

In [12]:
DELETE_UPLOADED_VIDEOS = False

if DELETE_UPLOADED_VIDEOS:
    for item in videos:
        item["video"].delete()
        print("Deleted:", item["title"])
else:
    print("Skipping cleanup. Set DELETE_UPLOADED_VIDEOS = True to delete sample uploads.")

Skipping cleanup. Set DELETE_UPLOADED_VIDEOS = True to delete sample uploads.


## Conclusion

You built a simple video RAG workflow with VideoDB and LlamaIndex:

- `understand(spoken_words)` created transcript artifacts.
- Transcript windows became timestamped retrieval records.
- `video.index()` made those records semantically searchable.
- A custom LlamaIndex retriever adapted VideoDB search results into `TextNode` objects.
- Retrieved nodes were compiled into a playable video stream.

## Further Resources

- [Understanding Artifacts](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/indexing-pipelines/understanding-artifacts)
- [Create an Index](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/indexing-pipelines/create-an-index)
- [Search and Retrieval](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/search-and-retrieval/natural-language-query)
